In [2]:
import sys, time
sys.path.append("../src")        # src 폴더를 import 경로에 추가

import paths                     # HF_HOME 설정 (가장 먼저)
import torch, tiktoken
from dataset import get_batch

enc = tiktoken.get_encoding("gpt2")
device = "cuda"

get_batch("train", 16, 256, device)      # 첫 호출: 파일 로딩 (측정에서 제외)

t0 = time.perf_counter()
for _ in range(100):
    get_batch("train", 16, 256, device)
torch.cuda.synchronize()
print(f"배치 1개 뽑는 시간: {(time.perf_counter() - t0) / 100 * 1000:.2f} ms")

배치 1개 뽑는 시간: 0.29 ms


In [3]:
from model import GPT, GPTConfig

torch.manual_seed(42)
model = GPT(GPTConfig()).to(device)          # 파라미터를 GPU로
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

x, y = get_batch("train", 16, 256, device)   # (16, 256) 배치 하나 고정

for step in range(100):
    logits, loss = model(x, y)               # logits (B, T, V), loss 스칼라
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

step   0  loss 10.8849
step  10  loss 8.7178
step  20  loss 7.1883
step  30  loss 5.9477
step  40  loss 5.1796
step  50  loss 4.4437
step  60  loss 3.7795
step  70  loss 3.0643
step  80  loss 2.3339
step  90  loss 1.7811


In [4]:
torch.manual_seed(42)
model = GPT(GPTConfig()).to(device)          # 새 모델 (셀 2에서 외운 상태 버림)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

B, T, steps = 16, 256, 300
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()

for step in range(steps):
    x, y = get_batch("train", B, T, device)
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f"\n스텝당 {dt / steps * 1000:.0f} ms")
print(f"처리량 {steps * B * T / dt:,.0f} tokens/s")
print(f"최대 GPU 메모리 {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

step   0  loss 10.8882
step  50  loss 5.5401
step 100  loss 4.7196
step 150  loss 4.4211
step 200  loss 3.9961
step 250  loss 3.8204

스텝당 136 ms
처리량 30,133 tokens/s
최대 GPU 메모리 4.29 GB


In [5]:
torch.set_float32_matmul_precision("high")   # 남은 fp32 행렬곱도 TF32 텐서 코어 사용

def bench_train(B, amp, steps=50, warmup=5, T=256):
    torch.manual_seed(42)
    model = GPT(GPTConfig()).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for step in range(warmup + steps):
        if step == warmup:                    # 워밍업 스텝은 측정에서 제외
            torch.cuda.synchronize()
            t0 = time.perf_counter()
        x, y = get_batch("train", B, T, device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=amp):
            logits, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    mem = torch.cuda.max_memory_allocated() / 1e9
    name = "bf16" if amp else "fp32"
    print(f"{name} B={B:3d}: {dt / steps * 1000:6.0f} ms/step  "
          f"{steps * B * T / dt:9,.0f} tok/s  최대 메모리 {mem:.2f} GB")
    del model, opt, logits, loss
    torch.cuda.empty_cache()

for B, amp in [(16, False), (16, True), (32, True), (48, True)]:
    try:
        bench_train(B, amp)
    except torch.cuda.OutOfMemoryError:
        print(f"{'bf16' if amp else 'fp32'} B={B:3d}: OOM")
        torch.cuda.empty_cache()

fp32 B= 16:    101 ms/step     40,718 tok/s  최대 메모리 5.60 GB
bf16 B= 16:     72 ms/step     56,536 tok/s  최대 메모리 4.58 GB
bf16 B= 32:    236 ms/step     34,707 tok/s  최대 메모리 7.39 GB
bf16 B= 48:   1205 ms/step     10,201 tok/s  최대 메모리 10.22 GB


In [6]:
import math

max_lr, min_lr = 6e-4, 6e-5
warmup_steps, max_steps = 200, 9000

def get_lr(step):
    if step < warmup_steps:                                   # ① warmup: 선형 증가
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:                                     # ③ 끝난 뒤: 최소값 유지
        return min_lr
    ratio = (step - warmup_steps) / (max_steps - warmup_steps)    # 0 → 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))               # 1 → 0 (코사인 모양)
    return min_lr + coeff * (max_lr - min_lr)                 # ② cosine decay

for s in [0, 100, 199, 200, 2000, 4600, 7000, 8999]:
    print(f"step {s:5d}: lr {get_lr(s):.2e}")

step     0: lr 3.00e-06
step   100: lr 3.03e-04
step   199: lr 6.00e-04
step   200: lr 6.00e-04
step  2000: lr 5.46e-04
step  4600: lr 3.30e-04
step  7000: lr 1.26e-04
step  8999: lr 6.00e-05


In [7]:
torch.manual_seed(42)
model = GPT(GPTConfig()).to(device)

# weight decay: 2차원 이상(행렬)만 적용
decay    = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=max_lr, betas=(0.9, 0.95), fused=True,      # fused: GPU에서 업데이트를 한 번에 처리해 빠름
)

B, T, K = 16, 256, 8
for step in range(30):
    t0 = time.perf_counter()

    # 1. 이번 스텝의 학습률 설정
    lr = get_lr(step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    # 2. K번 모으기 (gradient accumulation)
    optimizer.zero_grad(set_to_none=True)
    loss_accum = 0.0
    for micro in range(K):
        x, y = get_batch("train", B, T, device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits, loss = model(x, y)
        loss = loss / K
        loss_accum += loss.detach()          # 출력용 평균 loss (계산 그래프와 분리)
        loss.backward()

    # 3. clipping → 4. 업데이트
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    print(f"step {step:3d} | loss {loss_accum.item():.4f} | lr {lr:.2e} | "
          f"norm {norm.item():6.2f} | {dt * 1000:.0f} ms")

step   0 | loss 10.8833 | lr 3.00e-06 | norm   7.97 | 841 ms
step   1 | loss 10.8449 | lr 6.00e-06 | norm   8.09 | 517 ms
step   2 | loss 10.7805 | lr 9.00e-06 | norm   7.78 | 515 ms
step   3 | loss 10.6761 | lr 1.20e-05 | norm   7.04 | 514 ms
step   4 | loss 10.5565 | lr 1.50e-05 | norm   6.12 | 515 ms
step   5 | loss 10.4567 | lr 1.80e-05 | norm   5.16 | 515 ms
step   6 | loss 10.3391 | lr 2.10e-05 | norm   4.61 | 514 ms
step   7 | loss 10.2633 | lr 2.40e-05 | norm   4.05 | 515 ms
step   8 | loss 10.1679 | lr 2.70e-05 | norm   3.57 | 516 ms
step   9 | loss 10.0922 | lr 3.00e-05 | norm   3.22 | 516 ms
step  10 | loss 10.0300 | lr 3.30e-05 | norm   2.96 | 515 ms
step  11 | loss 9.9814 | lr 3.60e-05 | norm   2.81 | 515 ms
step  12 | loss 9.9289 | lr 3.90e-05 | norm   2.69 | 517 ms
step  13 | loss 9.8752 | lr 4.20e-05 | norm   2.63 | 515 ms
step  14 | loss 9.8172 | lr 4.50e-05 | norm   2.59 | 516 ms
step  15 | loss 9.7833 | lr 4.80e-05 | norm   2.56 | 516 ms
step  16 | loss 9.7351 | lr 5